# F7-kernels-convex-optimization — Practice p07

**Type:** constrained coding · **Difficulty:** core · **Concepts:** positive-semidefinite-matrices, kernel-validity

Implement `negative_eigen_witness(K)`.

Input contract: `K` is a finite real-numeric NumPy array of shape `(n,n)` with `n >= 1`, symmetric under `np.allclose(K, K.T, atol=ATOL, rtol=RTOL)`. Reject any other input with `ValueError`; do not mutate it.

If the symmetric average `(K + K.T) / 2` has no eigenvalue strictly below `-ATOL`, raise `ValueError`. Otherwise return `(lam, v)`, where `lam` is a finite plain `float` equal to the smallest eigenvalue and `v` is a finite floating NumPy array of shape `(n,)` with unit norm and

Writing $S=(K+K^T)/2$, require

$$Sv=\lambda v,\qquad v^TKv=v^TSv=\lambda<-\texttt{ATOL}.$$

Use `ATOL = 1e-10`, `RTOL = 0.0`. The vector's sign is irrelevant. A returned negative-energy pair is a constructive finite certificate that this matrix is not PSD and therefore refutes any kernel candidate that produced it.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

def negative_eigen_witness(K):
    """Return the least eigenpair when it certifies negative energy."""
    if (
        not isinstance(K, np.ndarray)
        or K.ndim != 2
        or K.shape[0] < 1
        or K.shape[0] != K.shape[1]
        or not np.issubdtype(K.dtype, np.number)
        or np.iscomplexobj(K)
        or not np.isfinite(K).all()
        or not np.allclose(K, K.T, atol=ATOL, rtol=RTOL)
    ):
        raise ValueError("K must be a finite real-numeric symmetric square array")

    K_float = K.astype(float, copy=False)
    S = (K_float + K_float.T) / 2.0
    eigenvalues, eigenvectors = np.linalg.eigh(S)
    lam = float(eigenvalues[0])
    if lam >= -ATOL:
        raise ValueError("K has no eigenvalue strictly below -ATOL")
    v = np.asarray(eigenvectors[:, 0], dtype=float)
    return lam, v

## Immutable contract check — do not edit

The public fixtures use different spectra and orientations. They verify the minimum eigenvalue, unit eigenvector equation, negative energy, rejection boundary, types, and non-mutation without fixing the eigenvector sign.

In [ ]:
_fixtures_p07 = (
    np.array([[-1.0]]),
    np.array([[-2.0, 0.5 * ATOL], [0.0, 1.0]]),
    np.array([[0.0, 2.0], [2.0, 0.0]]),
    np.diag(np.array([-4.0, -1.0, 3.0])),
    np.array([[1.0, 3.0], [3.0, 1.0]]),
)
for _K_p07 in _fixtures_p07:
    _before_p07 = _K_p07.copy()
    _out_p07 = negative_eigen_witness(_K_p07)
    assert type(_out_p07) is tuple and len(_out_p07) == 2
    _lam_p07, _v_p07 = _out_p07
    assert np.array_equal(_K_p07, _before_p07)
    assert type(_lam_p07) is float and np.isfinite(_lam_p07)
    assert isinstance(_v_p07, np.ndarray) and _v_p07.shape == (_K_p07.shape[0],)
    assert np.issubdtype(_v_p07.dtype, np.floating) and np.isfinite(_v_p07).all()
    _S_p07 = (_K_p07 + _K_p07.T) / 2.0
    _expected_lam_p07 = float(np.linalg.eigvalsh(_S_p07)[0])
    assert np.isclose(_lam_p07, _expected_lam_p07, atol=ATOL, rtol=RTOL)
    assert np.isclose(np.linalg.norm(_v_p07), 1.0, atol=ATOL, rtol=RTOL)
    assert np.allclose(_S_p07 @ _v_p07, _lam_p07 * _v_p07, atol=ATOL, rtol=RTOL)
    assert np.isclose(float(_v_p07 @ _S_p07 @ _v_p07), _lam_p07, atol=ATOL, rtol=RTOL)
    assert np.isclose(float(_v_p07 @ _K_p07 @ _v_p07), _lam_p07, atol=ATOL, rtol=RTOL)
    assert _lam_p07 < -ATOL

_invalid_p07 = (
    [[-1.0]],
    np.array([[-1.0, 0.0], [0.0, 1.0]])[:, :1],
    np.array([[0.0, 1.0], [0.0, 0.0]]),
    np.eye(2),
    np.diag(np.array([-0.5 * ATOL, 1.0])),
    np.array([[np.nan]]),
    np.array([[-1.0 + 0.0j]]),
)
for _bad_p07 in _invalid_p07:
    try:
        negative_eigen_witness(_bad_p07)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid or nonnegative K must raise ValueError")

### Solution reasoning

Averaging removes any tolerated skew-symmetric roundoff while preserving every quadratic form, since \(v^T(K-K^T)v=0\). For symmetric \(S\), the eigensolver returns ascending eigenvalues and orthonormal eigenvectors. Therefore the first pair is normalized, obeys \(Sv=\lambda v\), and minimizes the Rayleigh quotient. Returning it only when \(\lambda<-\texttt{ATOL}\) gives the requested constructive negative-energy certificate.

### Answer check

In [ ]:
_K_answer_p07 = np.array([[0.0, 2.0], [2.0, 0.0]])
_lam_answer_p07, _v_answer_p07 = negative_eigen_witness(_K_answer_p07)
assert np.isclose(_lam_answer_p07, -2.0, atol=ATOL, rtol=RTOL)
assert np.isclose(np.linalg.norm(_v_answer_p07), 1.0, atol=ATOL, rtol=RTOL)
assert np.isclose(
    _v_answer_p07 @ _K_answer_p07 @ _v_answer_p07,
    _lam_answer_p07,
    atol=ATOL,
    rtol=RTOL,
)